In [1]:
import os
print(os.getcwd())

C:\Users\silpi\Documents\encrypted-traffic-classifier-xai\notebooks


In [2]:
import pandas as pd
import numpy as np
import json
from sklearn.preprocessing import StandardScaler

# Load the pre-split data
X_train = pd.read_csv('../data/processed/X_train.csv')
X_test = pd.read_csv('../data/processed/X_test.csv')
y_train = np.load('../data/processed/y_train.npy')
y_test = np.load('../data/processed/y_test.npy')

with open('../data/processed/label_mapping.json') as f:
    label_mapping = json.load(f)

# Scale features (LSTM needs this, same as CNN)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train.values)
X_test_scaled = scaler.transform(X_test.values)

# Reshape for LSTM: (samples, timesteps, features)
# We treat each sample as 1 timestep with 23 features
X_train_reshaped = X_train_scaled.reshape(X_train_scaled.shape[0], 1, X_train_scaled.shape[1])
X_test_reshaped = X_test_scaled.reshape(X_test_scaled.shape[0], 1, X_test_scaled.shape[1])

print("X_train_reshaped shape:", X_train_reshaped.shape)
print("X_test_reshaped shape:", X_test_reshaped.shape)

X_train_reshaped shape: (47764, 1, 23)
X_test_reshaped shape: (11942, 1, 23)


In [3]:
import tensorflow as tf

y_train_onehot = tf.keras.utils.to_categorical(y_train)
y_test_onehot = tf.keras.utils.to_categorical(y_test)

print("y_train_onehot shape:", y_train_onehot.shape)
print("y_test_onehot shape:", y_test_onehot.shape)


C:\Users\silpi\anaconda3\envs\traffic-classifier\lib\site-packages\requests\__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


y_train_onehot shape: (47764, 14)
y_test_onehot shape: (11942, 14)


In [4]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

num_classes = y_train_onehot.shape[1]

model = Sequential([
    LSTM(64, input_shape=(1, 23), return_sequences=False),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dense(num_classes, activation='softmax')
])

model.summary()

C:\Users\silpi\anaconda3\envs\traffic-classifier\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape          ┃      Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│ lstm (LSTM)                   │ (None, 64)            │       22,528 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ dropout (Dropout)             │ (None, 64)            │            0 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ dense (Dense)                 │ (None, 64)            │        4,160 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ dense_1 (Dense)               │ (None, 14)            │          910 │
└───────────────────────────────┴───────────────────────┴──────────────┘

 Total params: 27,598 (107.80 KB)

 Trainable params: 27,598 (107.80 KB)

 Non-trainable params: 0 (0.00 B)

In [5]:
from tensorflow.keras.callbacks import EarlyStopping

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True)

history = model.fit(
    X_train_reshaped, y_train_onehot,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop]
)

Epoch 1/50
1195/1195 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.3249 - loss: 1.9993 - val_accuracy: 0.3715 - val_loss: 1.8046
Epoch 2/50
1195/1195 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.3753 - loss: 1.7935 - val_accuracy: 0.4109 - val_loss: 1.7183
Epoch 3/50
1195/1195 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.3995 - loss: 1.7245 - val_accuracy: 0.4377 - val_loss: 1.6493
Epoch 4/50
1195/1195 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.4132 - loss: 1.6799 - val_accuracy: 0.4242 - val_loss: 1.6080
Epoch 5/50
1195/1195 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.4278 - loss: 1.6355 - val_accuracy: 0.4402 - val_loss: 1.5704
Epoch 6/50
1195/1195 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.4350 - loss: 1.6004 - val_accuracy: 0.4587 - val_loss: 1.5319
Epoch 7/50
1195/1195 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.4440 - loss: 1.5681 - val_accuracy: 0.4628 - val_loss: 1.4989
Epoch 8/50
1195/1195 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.4547 - loss: 1.5423 - 

In [6]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
import numpy as np

y_pred_probs = model.predict(X_test_reshaped)
y_pred = np.argmax(y_pred_probs, axis=1)

id_to_label = {v: k for k, v in label_mapping.items()}
target_names = [id_to_label[i] for i in sorted(id_to_label.keys())]

accuracy = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average='macro')
weighted_f1 = f1_score(y_test, y_pred, average='weighted')

print(f"Test Accuracy: {accuracy:.4f}")
print(f"Macro F1: {macro_f1:.4f}")
print(f"Weighted F1: {weighted_f1:.4f}\n")

print(classification_report(y_test, y_pred, target_names=target_names))

cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)


374/374 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Test Accuracy: 0.5753
Macro F1: 0.5175
Weighted F1: 0.5616

               precision    recall  f1-score   support

     BROWSING       0.53      0.77      0.63      2000
         CHAT       0.46      0.29      0.36       501
           FT       0.66      0.44      0.53       795
         MAIL       0.37      0.13      0.19       273
          P2P       0.52      0.67      0.59       800
    STREAMING       0.48      0.37      0.42       257
         VOIP       0.71      0.90      0.79      1297
 VPN-BROWSING       0.53      0.53      0.53      2000
     VPN-CHAT       0.49      0.27      0.35       568
       VPN-FT       0.60      0.44      0.51       941
     VPN-MAIL       0.65      0.70      0.68       489
      VPN-P2P       0.49      0.55      0.52       683
VPN-STREAMING       0.70      0.50      0.58       223
     VPN-VOIP       0.72      0.48      0.58      1115

     accuracy                           0.58     11942
    macro avg    

In [7]:
import os
os.makedirs('../models', exist_ok=True)
model.save('../models/lstm_model.h5')
print("Model saved")

Model saved
